In [ ]:

import torch
import os, random
import numpy as np
from transformers import set_seed

SEED = 159753


# 1) Python, NumPy, PyTorch
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# 2) HF helper
set_seed(SEED)

In [ ]:
save_path = "/run/media/victor/pessoal/mestrado/codigo/model/V2_subfigures_model"

In [ ]:
from transformers import pipeline

classifier = pipeline("text-classification", model=save_path, truncation=True)

In [ ]:
a = classifier("Low power magnification H&E analysis of fixed spleen.(A–D) Spleen fixed in room temperature formalin for 0, 2, 4, or 24 hr. (E) Spleen fixed with 2+2 protocol. Images are 100× magnification.")

In [ ]:
a[0]

In [ ]:
import os
import glob

folder_path = '/run/media/victor/pessoal/mestrado/codigo/datasets/V2_leio_path_single'


images_paths = glob.glob(os.path.join(folder_path, '**', '*.*'), recursive=True)
images_paths = [path for path in images_paths if path.lower().endswith(('.jpg', '.jpeg', '.png'))]

In [ ]:
new_ids = [path.split("/")[-1].replace(".jpg", "") for path in images_paths]

In [ ]:
import pandas as pd

art_caption_df =  pd.read_csv("/run/media/victor/pessoal/mestrado/codigo/database_scripts/V2_leiomioma_animal_clean.csv")

In [ ]:
art_caption_df["new_id"] = art_caption_df["article_id"] + "-" + art_caption_df["id"]

In [ ]:
hist_art_caption_df = art_caption_df[art_caption_df["new_id"].isin(new_ids)].copy()

In [ ]:
hist_art_caption_df

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

class CaptionDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        caption = self.dataframe.iloc[idx]["caption"]
        return caption

# Create the dataset and dataloader
dataset = CaptionDataset(hist_art_caption_df)
dataloader = DataLoader(dataset, batch_size=32, shuffle=False)

results = []
scores = []
for batch in tqdm(dataloader, desc="Processing captions"):
    batch_results = classifier(batch)
    results.extend([res["label"] for res in batch_results])
    scores.extend([res["score"] for res in batch_results])


In [ ]:
hist_art_caption_df["result"] = results
hist_art_caption_df["scores"] = scores

In [ ]:
hist_art_caption_df

In [ ]:
hist_art_caption_df[hist_art_caption_df["id"] == "pone.0054138.g007"]

In [ ]:
hist_art_caption_df[hist_art_caption_df["result"] == "SINGLE"]

In [ ]:
hist_art_caption_df[hist_art_caption_df["result"] == "MULTI"]["caption"].to_csv("V2_multi_captions.csv", index=False)

In [ ]:
hist_art_caption_df[hist_art_caption_df["result"] == "SINGLE"].drop(columns=[ "new_id", "result", "scores"]).to_csv("V2_leiomyoma_cap_only_single.csv", index=False)

In [ ]:
hist_art_caption_df["License"].unique()

In [ ]:
import shutil

single_df = hist_art_caption_df[(hist_art_caption_df["result"] == "SINGLE")]

with tqdm(total=len(single_df)) as pbar:
    for _, row in single_df.iterrows():
        # print(row["image_path"].split('/')[-1])
        shutil.copyfile(row["image_path"], f'/run/media/victor/pessoal/mestrado/codigo/datasets/V2_leio_path_only_single/{row["article_id"]}-{row["image_path"].split('/')[-1]}')
        pbar.update(1)